# Automatic Differentiation
Fare tutti i calcoli visti nel capitolo precendete potrebbe portarci facilemente a fare errori, per questo nei moderni framework di deep learning come PyTorch è già implementata l'automatic differentiation (autograd). Passando i dati tra i vari layer (funzioni) il framework costruisce un grafo computazionale che tiene traccia di ogni valore in funzione degli altri valori. Per calcolare le derivate, autograd parte dal risultato e torna indietro applicando la chain rule, l'algoritmo che applica la chain rule si chiama backpropagation.

In [1]:
import torch

## A Simple Function
Supponiamo di voler derivare la funzione $y=2\pmb{x}^T\pmb{x}$ rispetto al vettore $\pmb{x}$. Per prima cosa diamo dei valori iniziali ad $x$.

In [2]:
x = torch.arange(4.0)
x

tensor([0., 1., 2., 3.])

Prima di calcolare il gradiente di $y$ rispetto ad $x$ abbiamo bisogno di una variabile dove memorizzarlo. Dato che il deep learning richiede calcoli di derivate concatenati molte volte non vogliamo sempre allocare nuova memoria.

In [3]:
x.requires_grad_(True)
x.grad # Il gradient sarà None di default

# Possiamo anche farlo da subito con
# x = torch.arange(4.0, requires_grad=True)

Adesso possiamo calcolare la funzione $x$ e assegnare il risultato ad $y$

In [4]:
y = 2 * torch.dot(x, x)
y

tensor(28., grad_fn=<MulBackward0>)

Adesso possiamo calcolare il gradiente di $y$ chiamando il metodo `backward` e poi accedere a questo tramite l'attributo `grad`

In [5]:
y.backward()
x.grad

tensor([ 0.,  4.,  8., 12.])

Infatti nel capitolo precedente abbiamo visto che la derivata di $y=2x^Tx$ rispetto ad $x$ è $4x$. Possiamo verificarlo

In [6]:
x.grad == 4 * x

tensor([True, True, True, True])

## Backward for Non-Scalar Variables
Il gradiente va bene per funzioni che danno in output uno scalare (un singolo numero). Se la nostra funzione $y$ restituisce un vettore allora il modo migliore per rappresentare il suo "gradiente" è la matrice Jacobiana, ogni riga di questa matrice rappresenta appunto il gradiente di ogni componente di $y$ rispetto ad $x$.

Di solito però questa matrice non viene utilizzata ma ci interessa di più sommare tutti questi gradienti per ottenere l'errore commesso dalla rete.

In PyTorch ad esempio chiamare la funzione _backward_ su un non scalare produce un errore, a meno che non diciamo a PyTorch come ridurre l'oggetto in uno scalare. In modo più formale dobbiamo fornire un vettore $\pmb{v}$ tale che la funzione di _backward_ computi $\pmb{v}^T\partial_{\pmb{x}}\pmb{y}$ invece che $\partial_{\pmb{x}}\pmb{y}$. Questo campo da aggiungere $\pmb{v}$ si chiama gradient.

In [9]:
x.grad.zero_()
y = x * x
y.backward(gradient=torch.ones(len(y)))
# Stessa cosa di
# y.sum().backward()
x.grad

tensor([0., 2., 4., 6.])

### Riassuntino
Ho fatto abbastanza fatica con tutta la roba precedente quindi me le riassumo un pochino.
Il gradiente entra in gioco quando abbiamo una funzione che prende tanti numeri in input ma restituisce un solo numero in output, ad esempio un'immagine (vettore x) in input e restituisce l'errore totale finale.
Chiamare _backward_ ci permette di ottenere un vettore della stessa identica forma di x che ci dice come aggiornare i pesi.

Se la nostra funzione però restituisce tanti numeri ovvero un altro vettore y allora usiamo una matrice Jacobiana. Nel Deep Learning questo accade perché non analizziamo mai immagine per immagine ma batch di immagini (32 ad esempio) e quindi otteniamo un vettore di 32 errori separati. Calcolare la Jacobiana però consumerebbe tantissima memoria e in realtà non ci interessa quasi mai, spesso vogliamo soltanto l'errore complessivo della batch.
Se noi passiamo a gradient un vettore di soli 1, quando PyTorch calcola le derivate usa questo vettore per moltiplicare le righe della Jacobiana comprimendola in un solo vettore finale

## Detaching Computation
In alcuni casi potremmo voler spostare alcuni calcoli fuori dal grafo computazionale, ad esempio vogliamo usare l'input per creare dei dati intermedi ausiliari e dei quali non vogliamo calcolare un gradiente.
In questi casi dobbiamo fare il _detach_.
Vediamo un esempio dove abbiamo $z = x * y$ e $y = x * x$ ma ci vogliamo interessare sull'influenza di $x$ su $z$ invece che su $y$. Possiamo creare una nuova variabile $u$ che prende lo stesso valore di $y$ ma la cui provenienza viene cancellata (l'operazione da cui è stata generata).

In [10]:
x.grad.zero_()
y = x * x
u = y.detach()
z = u * x

z.sum().backward()
x.grad == u

tensor([True, True, True, True])

Questo perché appunto $u$ è diventata una costante all'interno di $z$, non è più una funzione di $x$.
Da notare però che anche se abbiamo scollegato $y$ dal grafo, quello collegato fino ad $y$ esiste ancora e possiamo quindi calcolare il gradiente di $y$ rispetto ad $x$.

In [11]:
x.grad.zero_()
y.sum().backward()
x.grad == 2 * x

tensor([True, True, True, True])

## Gradients and Python Control Flow
Fino ad ora abbiamo visto soltanto casi dove la funzione viene sempre calcolata allo stesso modo, possiamo però effettuare calcoli in base ad altre variabili, quindi utilizzando semplici if di Python e comunque riuscire a calcolare i gradienti.

In [12]:
def f(a):
    b = a * 2
    while b.norm() < 1000:
        b = b * 2
    if b.sum() > 0:
        c = b
    else:
        c = 100 * b
    return c

Adesso possiamo chiamare questa funzione con un valore randomico e quindi non sappiamo che strada prenderà il grafo, ma possiamo comunque percorrerlo.

In [13]:
a = torch.randn(size=(), requires_grad=True)
d = f(a)
d.backward()

In [14]:
a.grad == d / a

tensor(True)